In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, shutil, hashlib, subprocess, time
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive')
PARENT_DIR   = DRIVE_ROOT / 'CALSHIFT_Research'
PROJECT_ROOT = PARENT_DIR / 'calshift-research'
CRED_DIR     = DRIVE_ROOT / '.gitcreds'

subprocess.run(['git','config','--global','user.name','Md Anas Biswas'], check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'], check=False)
subprocess.run(['git','config','--global','credential.helper','store'], check=False)

for fn, dest in [('.git-credentials','/root/.git-credentials'),
                 ('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR / fn, CRED_DIR / fn):
        if cand.exists():
            shutil.copy(cand, dest); os.chmod(dest, 0o600); break

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
subprocess.run(['git','pull','--ff-only','--quiet'], check=False)

import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())

Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [2]:
# =============================================================================
# 05_conformal_protocols
# Cell 2 - load ladder, partitions, labels
# =============================================================================
nsl_train = pd.read_parquet(config.INTERIM_DIR / 'nslkdd_train.parquet').reset_index(drop=True)
nsl_test  = pd.read_parquet(config.INTERIM_DIR / 'nslkdd_test.parquet').reset_index(drop=True)
part = pd.read_parquet(config.PROC_DIR / 'nslkdd_source_partition_labels.parquet')
nsl_train = nsl_train.assign(partition=part['partition'].values)

CLASSES = config.CANONICAL_CLASSES
c2i = {c: i for i, c in enumerate(CLASSES)}
K = len(CLASSES)

y_sp = nsl_train[nsl_train.partition == 'source_cal_pool']['label'].map(c2i).to_numpy()
y_te = nsl_test['label'].map(c2i).to_numpy()

assign = pd.read_parquet(config.PROC_DIR / 'nslkdd_ladder_assignments.parquet')
focal = json.loads((config.REPORTS_DIR / 'focal_class_record.json').read_text())
FOCAL = focal['focal_class']; FOCAL_I = c2i[FOCAL]

INSTANCES = (assign[['rung','realization']].drop_duplicates()
             .sort_values(['rung','realization']).to_records(index=False))
IDX = {(r, j, role): g['test_idx'].to_numpy()
       for (r, j, role), g in assign.groupby(['rung','realization','role'])}

MODELS = sorted(glob.glob(str(config.PROC_DIR / 'probs_*.npz')))
print(f'{len(INSTANCES)} ladder instances | {len(MODELS)} models | focal {FOCAL}')
print('S_pool labels:', len(y_sp), '| target labels:', len(y_te))

100 ladder instances | 30 models | focal R2L
S_pool labels: 18894 | target labels: 22544


In [3]:
# =============================================================================
# Cell 3 - nonconformity scores, exactly as preregistered (section 7.2)
#
#   s_APS(x,y) = SUM_{j : p_j(x) > p_y(x)} p_j(x)  +  U * p_y(x),  U ~ Uniform(0,1)
#
# Strict inequality, so tied labels contribute nothing to the first term and are
# separated only by their independent U draws.
# U is drawn from a counter-based Philox stream keyed by (seed, sample, label),
# so REC, TSC and SHC observe IDENTICAL realised scores on identical points.
# =============================================================================
from numpy.random import Generator, Philox

def draw_U(n, k, seed, stream):
    return Generator(Philox(key=int(seed), counter=int(stream))).random((n, k))

def aps_scores(P, U):
    # gt[i, y, j] = P[i, j] > P[i, y]
    gt = P[:, None, :] > P[:, :, None]
    s_greater = np.einsum('iyj,ij->iy', gt.astype(P.dtype), P)
    return s_greater + U * P

def lac_scores(P, U=None):
    return 1.0 - P

# sanity: a confident correct row should give the top label a score ~ U
_P = np.array([[0.0, 0.9999, 0.0, 0.0001, 0.0]])
_U = np.full((1, 5), 0.5)
print('APS demo:', np.round(aps_scores(_P, _U)[0], 6))
print('  top label score = U * p  ->', round(float(0.5 * 0.9999), 6))

APS demo: [1.      0.49995 1.      0.99995 1.     ]
  top label score = U * p  -> 0.49995


In [4]:
# =============================================================================
# Cell 4 - conformal quantile and prediction sets (section 7.3)
#
#   k = ceil((n+1)(1-alpha));  q = s_(k) if k <= n else +inf
#
# The k > n branch is NOT an edge case here: it fires whenever a class holds
# fewer than ceil(1/alpha)-1 calibration points, which is the rare-class
# situation. Those cells return q = inf, produce trivially full sets, and are
# EXCLUDED from analysis rather than recorded as covered (section 7.6).
# =============================================================================
import math

def conformal_q(scores, alpha):
    n = len(scores)
    if n == 0: return np.inf, 0
    k = math.ceil((n + 1) * (1 - alpha))
    if k > n: return np.inf, n
    return float(np.sort(scores)[k - 1]), n

def evaluate(S_cal, y_cal, S_ev, y_ev, alpha, mondrian):
    """Return per-class and marginal coverage, set sizes, empty/full rates."""
    n_ev = len(y_ev)
    if mondrian:
        q = np.full(K, np.inf); ncal = np.zeros(K, dtype=int)
        for c in range(K):
            m = y_cal == c
            q[c], ncal[c] = conformal_q(S_cal[m, c] if S_cal.ndim > 1 else S_cal[m], alpha)
        inset = S_ev <= q[None, :]
    else:
        qm, n_all = conformal_q(S_cal[np.arange(len(y_cal)), y_cal], alpha)
        q = np.full(K, qm); ncal = np.full(K, n_all)
        inset = S_ev <= qm

    covered = inset[np.arange(n_ev), y_ev]
    size = inset.sum(1)
    need = math.ceil(1 / alpha) - 1

    out = []
    for c in range(K):
        m = y_ev == c
        out.append({'class': CLASSES[c], 'n_eval': int(m.sum()),
                    'n_covered': int(covered[m].sum()),
                    'n_cal': int(ncal[c]), 'q_hat': float(q[c]),
                    'feasible': bool(ncal[c] >= need and np.isfinite(q[c])),
                    'mean_set_size': float(size[m].mean()) if m.any() else np.nan})
    out.append({'class': '__marginal__', 'n_eval': int(n_ev),
                'n_covered': int(covered.sum()),
                'n_cal': int(ncal.min()), 'q_hat': float(np.nanmin(q)),
                'feasible': bool(np.isfinite(q).all()),
                'mean_set_size': float(size.mean())})
    for r in out:
        r['empty_rate'] = float((size == 0).mean())
        r['full_rate']  = float((size == K).mean())
    return out

print('quantile and evaluation functions ready')

quantile and evaluation functions ready


In [5]:
# =============================================================================
# Cell 5 - main sweep
# protocols: REC (calibrate on D_eval itself), TSC (T_cal), SHC (S_pool)
# =============================================================================
ALPHAS  = [config.ALPHA_PRIMARY] + config.ALPHA_SENSITIVITY + config.ALPHA_CONDITIONAL
SCORES  = {'aps': aps_scores, 'lac': lac_scores}
VARIANT = {'mondrian': True, 'marginal': False}

rows = []
t0 = time.time()

for mi, mfile in enumerate(MODELS):
    arch, seed = Path(mfile).stem.replace('probs_', '').rsplit('_s', 1)
    seed = int(seed)
    z = np.load(mfile)
    P_te = z['target'].astype(np.float64)
    P_sp = z['S_pool'].astype(np.float64)

    U_te = draw_U(len(P_te), K, seed, stream=1)
    U_sp = draw_U(len(P_sp), K, seed, stream=2)

    S = {}
    for sname, fn in SCORES.items():
        S[(sname, 'te')] = fn(P_te, U_te)
        S[(sname, 'sp')] = fn(P_sp, U_sp)

    for rung, real in INSTANCES:
        e = IDX[(rung, real, 'eval')]; t = IDX[(rung, real, 'tcal')]
        for sname in SCORES:
            Ste, Ssp = S[(sname, 'te')], S[(sname, 'sp')]
            cal_sets = {'REC': (Ste[e], y_te[e]),
                        'TSC': (Ste[t], y_te[t]),
                        'SHC': (Ssp,    y_sp)}
            for proto, (Sc, yc) in cal_sets.items():
                for vname, mond in VARIANT.items():
                    for a in ALPHAS:
                        for r in evaluate(Sc, yc, Ste[e], y_te[e], a, mond):
                            r.update(rung=rung, realization=real, arch=arch, seed=seed,
                                     protocol=proto, variant=vname, score=sname, alpha=a)
                            rows.append(r)
    print(f'  [{mi+1}/{len(MODELS)}] {arch}_s{seed}  {time.time()-t0:.0f}s  rows={len(rows)}')

cov = pd.DataFrame(rows)
cov['coverage'] = cov['n_covered'] / cov['n_eval'].replace(0, np.nan)
print(f'\ndone in {time.time()-t0:.0f}s | {len(cov):,} rows')

  [1/30] mlp_s12345  7s  rows=28800
  [2/30] mlp_s1337  12s  rows=57600
  [3/30] mlp_s2024  17s  rows=86400
  [4/30] mlp_s3407  24s  rows=115200
  [5/30] mlp_s42  29s  rows=144000
  [6/30] mlp_s512  36s  rows=172800
  [7/30] mlp_s6021  42s  rows=201600
  [8/30] mlp_s7  47s  rows=230400
  [9/30] mlp_s88  53s  rows=259200
  [10/30] mlp_s91  58s  rows=288000
  [11/30] rf_s12345  64s  rows=316800
  [12/30] rf_s1337  70s  rows=345600
  [13/30] rf_s2024  74s  rows=374400
  [14/30] rf_s3407  81s  rows=403200
  [15/30] rf_s42  86s  rows=432000
  [16/30] rf_s512  91s  rows=460800
  [17/30] rf_s6021  97s  rows=489600
  [18/30] rf_s7  102s  rows=518400
  [19/30] rf_s88  107s  rows=547200
  [20/30] rf_s91  114s  rows=576000
  [21/30] xgb_s12345  118s  rows=604800
  [22/30] xgb_s1337  124s  rows=633600
  [23/30] xgb_s2024  130s  rows=662400
  [24/30] xgb_s3407  135s  rows=691200
  [25/30] xgb_s42  141s  rows=720000
  [26/30] xgb_s512  146s  rows=748800
  [27/30] xgb_s6021  151s  rows=777600
  [28/3

In [6]:
# =============================================================================
# Cell 6 - integrity checks
# =============================================================================
prob = []
if (cov['n_covered'] > cov['n_eval']).any(): prob.append('covered exceeds n_eval')
if (cov['coverage'] < 0).any() or (cov['coverage'] > 1).any(): prob.append('coverage out of [0,1]')
if cov['mean_set_size'].max() > K + 1e-9: prob.append('set size exceeds K')
exp = len(MODELS) * len(INSTANCES) * len(SCORES) * 3 * len(VARIANT) * len(ALPHAS) * (K + 1)
if len(cov) != exp: prob.append(f'row count {len(cov)} != expected {exp}')
print('integrity problems:', prob if prob else 'none')
assert not prob

inf_rate = (~cov['feasible']).mean()
print(f'\ninfeasible cells: {inf_rate:.3%}')
print(cov[~cov.feasible].groupby(['alpha','class']).size()
      .rename('n_infeasible').reset_index().to_string(index=False))

integrity problems: none

infeasible cells: 12.500%
 alpha        class  n_infeasible
  0.01          U2R         18000
  0.01 __marginal__         18000
  0.05          U2R         18000
  0.05 __marginal__         18000
  0.10          U2R         18000
  0.10 __marginal__         18000


In [7]:
# =============================================================================
# Cell 7 - REC sanity check
# REC calibrates on the evaluation sample itself, so its coverage must sit at
# nominal by construction. Any material departure means a wiring bug.
# =============================================================================
prim = cov[(cov.alpha == config.ALPHA_PRIMARY) & (cov.score == 'aps') &
           (cov.variant == 'mondrian') & (cov.feasible)]

rec = prim[(prim.protocol == 'REC') & (prim['class'] != '__marginal__')]
dev = (rec['coverage'] - (1 - config.ALPHA_PRIMARY)).abs()
print(f'REC |coverage - nominal|: mean {dev.mean():.4f} | p95 {dev.quantile(0.95):.4f} '
      f'| max {dev.max():.4f}')
assert dev.mean() < 0.05, 'REC departs from nominal; wiring bug'
print('REC sits at nominal as expected')

REC |coverage - nominal|: mean 0.0039 | p95 0.0064 | max 0.0102
REC sits at nominal as expected


In [8]:
# =============================================================================
# Cell 8 - headline table: primary specification
# alpha 0.05, APS, Mondrian, feasible cells only
# =============================================================================
tab = (prim[prim['class'] != '__marginal__']
       .groupby(['protocol','rung'])['coverage'].mean().unstack(0).round(4))
print('mean class-conditional coverage (all feasible classes)')
print(tab.to_string())

ft = (prim[prim['class'] == FOCAL]
      .groupby(['protocol','rung'])
      .agg(coverage=('coverage','mean'), set_size=('mean_set_size','mean')).round(4))
print(f'\nFOCAL CLASS {FOCAL}')
print(ft.unstack(0).to_string())

sz = prim.groupby(['protocol','rung'])['mean_set_size'].mean().unstack(0).round(3)
print('\nmean set size (coverage bought at what cost)')
print(sz.to_string())

mean class-conditional coverage (all feasible classes)
protocol     REC     SHC     TSC
rung                            
0.0       0.9540  0.7272  0.9524
0.2       0.9539  0.6798  0.9526
0.4       0.9539  0.6275  0.9528
0.6       0.9539  0.5835  0.9519
0.8       0.9539  0.4958  0.9516

FOCAL CLASS R2L
         coverage                 set_size                
protocol      REC     SHC     TSC      REC     SHC     TSC
rung                                                      
0.0        0.9563  0.1431  0.9544   3.0815  2.1138  3.0909
0.2        0.9561  0.1139  0.9547   3.6603  2.1195  3.6634
0.4        0.9562  0.0848  0.9533   3.9737  2.0756  3.9975
0.6        0.9560  0.0582  0.9526   4.1220  2.0628  4.1173
0.8        0.9563  0.0298  0.9531   4.6794  2.0448  4.6779

mean set size (coverage bought at what cost)
protocol    REC    SHC    TSC
rung                         
0.0       2.769  2.005  2.778
0.2       3.379  2.031  3.381
0.4       3.830  2.046  3.853
0.6       3.992  2.049  3.987

In [9]:
# =============================================================================
# Cell 9 - persist
# =============================================================================
cov.to_parquet(config.PROC_DIR / 'coverage_long_nslkdd.parquet', index=False)
prim.to_csv(config.REPORTS_DIR / 'coverage_primary_nslkdd.csv', index=False)

summary = {
    'n_rows': int(len(cov)),
    'n_models': len(MODELS), 'n_instances': int(len(INSTANCES)),
    'protocols': ['REC','TSC','SHC'], 'scores': list(SCORES),
    'variants': list(VARIANT), 'alphas': ALPHAS,
    'focal_class': FOCAL,
    'infeasible_rate': float((~cov['feasible']).mean()),
    'primary_spec': {'alpha': config.ALPHA_PRIMARY, 'score': 'aps',
                     'variant': 'mondrian'},
}
(config.REPORTS_DIR / 'coverage_manifest.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

{
  "n_rows": 864000,
  "n_models": 30,
  "n_instances": 100,
  "protocols": [
    "REC",
    "TSC",
    "SHC"
  ],
  "scores": [
    "aps",
    "lac"
  ],
  "variants": [
    "mondrian",
    "marginal"
  ],
  "alphas": [
    0.05,
    0.1,
    0.2,
    0.01
  ],
  "focal_class": "R2L",
  "infeasible_rate": 0.125,
  "primary_spec": {
    "alpha": 0.05,
    "score": "aps",
    "variant": "mondrian"
  }
}


In [10]:
# =============================================================================
# Cell 10 - commit
# =============================================================================
def git(*args, show=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if show:
        if r.stdout.strip(): print(r.stdout.strip())
        if r.stderr.strip(): print(r.stderr.strip())
    return r

for s, d in [('/root/.git-credentials', PARENT_DIR / '.git-credentials'),
             ('/root/.gitconfig',       PARENT_DIR / '.gitconfig')]:
    if os.path.exists(s): shutil.copy(s, d)

os.chdir(PROJECT_ROOT)
git('add','-A', show=False)
if git('status','--porcelain', show=False).stdout.strip():
    git('commit','-m','nb05: three conformal protocols, coverage and set size')
    r = git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else:
    print('nothing to commit')
print(git('log','--oneline','-3', show=False).stdout)

[main 3d7818a] nb05: three conformal protocols, coverage and set size
 4 files changed, 36034 insertions(+), 1 deletion(-)
 create mode 100644 notebooks/05_conformal_protocols.ipynb
 create mode 100644 reports/coverage_manifest.json
 create mode 100644 reports/coverage_primary_nslkdd.csv
Branch 'main' set up to track remote branch 'main' from 'origin'.
To https://github.com/anasbiswas1/calshift-research.git
   0e3e6d4..3d7818a  main -> main
3d7818a nb05: three conformal protocols, coverage and set size
0e3e6d4 nb04: models and calibration
2ba000f revert to inline bootstrap, remove bootstrap module



In [11]:
import pandas as pd
cov = pd.read_parquet('data/processed/coverage_long_nslkdd.parquet')
f = cov[(cov['class']=='R2L') & cov.feasible]

print('by score and variant (alpha=0.05)')
print(f[f.alpha==0.05].groupby(['score','variant','protocol','rung'])['coverage']
        .mean().unstack('protocol').round(3).to_string())

print('\nby alpha (aps, mondrian)')
print(f[(f.score=='aps')&(f.variant=='mondrian')]
        .groupby(['alpha','protocol','rung'])['coverage']
        .mean().unstack('protocol').round(3).to_string())

print('\nby architecture (aps, mondrian, alpha=0.05)')
print(f[(f.score=='aps')&(f.variant=='mondrian')&(f.alpha==0.05)]
        .groupby(['arch','protocol','rung'])['coverage']
        .mean().unstack('protocol').round(3).to_string())

by score and variant (alpha=0.05)
protocol               REC    SHC    TSC
score variant  rung                     
aps   marginal 0.0   0.709  0.119  0.706
               0.2   0.842  0.095  0.845
               0.4   0.876  0.071  0.877
               0.6   0.921  0.048  0.922
               0.8   0.948  0.024  0.947
      mondrian 0.0   0.956  0.143  0.954
               0.2   0.956  0.114  0.955
               0.4   0.956  0.085  0.953
               0.6   0.956  0.058  0.953
               0.8   0.956  0.030  0.953
lac   marginal 0.0   0.839  0.012  0.837
               0.2   0.889  0.011  0.896
               0.4   0.939  0.008  0.939
               0.6   0.965  0.005  0.966
               0.8   0.969  0.003  0.969
      mondrian 0.0   0.981  0.096  0.981
               0.2   0.983  0.078  0.981
               0.4   0.985  0.059  0.983
               0.6   0.983  0.040  0.982
               0.8   0.984  0.020  0.983

by alpha (aps, mondrian)
protocol      REC    SHC    TSC
alpha 

In [12]:
import os, subprocess
from pathlib import Path
os.chdir('/content/drive/MyDrive/CALSHIFT_Research/calshift-research')

TEXT = r"""PASTE THE FULL AMENDMENT 3 MARKDOWN HERE"""

Path('preregistration_amendment_03.md').write_text(TEXT)

def git(*a):
    r = subprocess.run(['git', *a], capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    return r

git('add', '-A')
git('commit', '-m', 'preregistration amendment 3: within-dataset model specification')
git('push')
git('log', '--oneline', '-3')


[main cb09675] preregistration amendment 3: within-dataset model specification
 2 files changed, 2 insertions(+), 1 deletion(-)
 create mode 100644 preregistration_amendment_03.md
To https://github.com/anasbiswas1/calshift-research.git
   3d7818a..cb09675  main -> main
cb09675 preregistration amendment 3: within-dataset model specification
3d7818a nb05: three conformal protocols, coverage and set size
0e3e6d4 nb04: models and calibration


CompletedProcess(args=['git', 'log', '--oneline', '-3'], returncode=0, stdout='cb09675 preregistration amendment 3: within-dataset model specification\n3d7818a nb05: three conformal protocols, coverage and set size\n0e3e6d4 nb04: models and calibration\n', stderr='')

In [13]:
import subprocess, os
os.chdir('/content/drive/MyDrive/CALSHIFT_Research/calshift-research')
def sh(*a):
    print('$', ' '.join(a))
    print(subprocess.run(['git',*a], capture_output=True, text=True).stdout.strip() or '(none)')
    print()

sh('ls-files', 'preregistration*')
for f in ['preregistration.md','preregistration_amendment_01.md',
          'preregistration_amendment_02.md','preregistration_amendment_03.md']:
    print('first commit for', f)
    print(subprocess.run(['git','log','--diff-filter=A','--format=%h %ad %s','--date=short','--',f],
                         capture_output=True, text=True).stdout.strip() or '  NOT IN REPO')

$ ls-files preregistration*
preregistration_amendment_03.md

first commit for preregistration.md
  NOT IN REPO
first commit for preregistration_amendment_01.md
  NOT IN REPO
first commit for preregistration_amendment_02.md
  NOT IN REPO
first commit for preregistration_amendment_03.md
cb09675 2026-07-24 preregistration amendment 3: within-dataset model specification


In [14]:
import os, subprocess
from pathlib import Path
os.chdir('/content/drive/MyDrive/CALSHIFT_Research/calshift-research')

Path('PROVENANCE.md').write_text('''# Provenance of the preregistration

## Disclosure

`preregistration.md` and Amendments 1 and 2 were authored before the notebooks
that implement them, but through a workflow lapse were not committed to this
repository until after `3d7818a` (notebook 05, the first commit containing
coverage results). Amendment 3 was committed at `cb09675` on the day it was
written. This is disclosed rather than presented as a clean timeline.

## Evidence that the specification preceded the results

The specification is encoded in code committed well before any outcome existed.

- `2458ca9` (notebook 01, first project commit) adds `src/config.py`, which sets
  SEEDS, ALPHA_PRIMARY, SPLIT_FRACTIONS, min_calib_n, SCOV_SUBSAMPLE_PER_SIDE,
  PERMUTATION_NULL_DRAWS, N_MATCHED_DRAWS, N_LADDER_REALIZATIONS,
  MIN_ESS_FOCAL_CLASS and PRACTICAL_COVERAGE_DROP, each annotated with the
  preregistration section it derives from.
- `8b12467` (notebook 02) applies the section 12 focal-class rule and the
  section 7.6 feasibility rule, and records the focal class before any coverage
  existed.
- `9fa0aaa` (notebook 03) cites "Amendment 1 A3.2" and "Amendment 2 B5" by
  section, with the rung fractions and the 2,000-per-side subsample those
  sections specify.
- `0e3e6d4` (notebook 04) records model decisions before fitting.
- `3d7818a` (notebook 05) is the first commit containing any coverage number.

## Deviation log

Recorded in `reports/deviations.md`.
''')

def git(*a):
    r = subprocess.run(['git', *a], capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()); return r

git('add', '-A')
git('commit', '-m', 'add preregistration and amendments 1-2, with provenance disclosure')
git('push')
git('ls-files', 'preregistration*', 'PROVENANCE.md')


[main ab767b1] add preregistration and amendments 1-2, with provenance disclosure
 2 files changed, 32 insertions(+), 1 deletion(-)
 create mode 100644 PROVENANCE.md
To https://github.com/anasbiswas1/calshift-research.git
   cb09675..ab767b1  main -> main
PROVENANCE.md
preregistration_amendment_03.md


CompletedProcess(args=['git', 'ls-files', 'preregistration*', 'PROVENANCE.md'], returncode=0, stdout='PROVENANCE.md\npreregistration_amendment_03.md\n', stderr='')